# 12. Pretests: stationarity and the Granger pretest

This notebook walks through the two checks that run before any fit:

| Function | Returns | Question |
|---|---|---|
| `check_stationarity` | `StationarityResult` | Which series trend, and where does each one go? |
| `granger_pretest` | `GrangerPretestResult` | Which series help predict which, before the rank is known? |

The third pretest, `select_coint_rank`, is in notebook 10. All three run on the data alone, in seconds, with no sampling.

The data is the simulated brand system from notebook 11. Its truth is known, so each verdict can be checked.

In [1]:
from __future__ import annotations

import pandas as pd

from bayesian_vecm import (
    GrangerPretestResult,
    StationarityResult,
    check_stationarity,
    granger_pretest,
    simulate_brand_data,
)

data = simulate_brand_data()

## 1. `check_stationarity`: which series trend?

### What it does

A VECM needs the order of integration of every input:

- **I(1)** series trend and become stationary after one difference. They go in `endog`, where they can share long-run relations.
- **I(0)** series return to a fixed level. They go in `exog`, where they act on short-run growth.
- **I(2)** series still trend after one difference. Difference them first.

`check_stationarity` runs the Augmented Dickey-Fuller (ADF) test twice per column: on the first difference and on the levels. The ADF null is a unit root, so a small p-value means stationary.

The difference is read first (Dickey and Pantula 1987). A series with two unit roots can reject on the levels by chance. The levels test only counts once the difference is known to be stationary.

### Why the model needs it

Notebook 01 showed that the VECM's left-hand side is $\Delta y_t$. A stationary series in `endog` has no trend to share, so it distorts the rank. A trending series in `exog` cannot explain stationary growth. Getting the roles right is the first step of the workflow.

In [2]:
both = pd.concat([data.endog, data.exog], axis=1)
result = check_stationarity(both)
assert isinstance(result, StationarityResult)
print(result)

Stationarity check (ADF test, 5% level)
  regression = 'c'

  variable             p (levels)   p (diff)  order  verdict
  -----------------------------------------------------------------------------------------------
  organic_sales             0.774      0.000   I(1)  trending: endog, or exog after .diff()
  brand_awareness           0.741      0.000   I(1)  trending: endog, or exog after .diff()
  brand_consideration       0.678      0.000   I(1)  trending: endog, or exog after .diff()
  brand_spend               0.000      0.000   I(0)  stationary: use as exog
  pr_reach                  0.000      0.000   I(0)  stationary: use as exog
  sponsorship_reach         0.000      0.000   I(0)  stationary: use as exog
  interest_rate             0.000      0.000   I(0)  stationary: use as exog
  consumer_confidence       0.000      0.000   I(0)  stationary: use as exog


The verdicts match the truth: the three brand metrics are I(1) and the five drivers are I(0).

The result also holds the table as a `DataFrame`, and two lists that drop straight into the model call:

In [3]:
print("endog:", result.endog)
print("exog: ", result.exog)
result.table.round(3)

endog: ['organic_sales', 'brand_awareness', 'brand_consideration']
exog:  ['brand_spend', 'pr_reach', 'sponsorship_reach', 'interest_rate', 'consumer_confidence']


,variable,p_level,p_diff,order,role
0,organic_sales,0.774,0.0,I(1),endog
1,brand_awareness,0.741,0.0,I(1),endog
2,brand_consideration,0.678,0.0,I(1),endog
3,brand_spend,0.000,0.0,I(0),exog
4,pr_reach,0.000,0.0,I(0),exog
5,sponsorship_reach,0.000,0.0,I(0),exog
6,interest_rate,0.000,0.0,I(0),exog
7,consumer_confidence,0.000,0.0,I(0),exog


### A trending driver

With `exog_unit_root=("interest_rate",)` the interest rate is a random walk (notebook 11 §4). The check flags it as I(1). The verdict says where it can go: `endog` if it shares a long-run relation with the brand series, or `exog` after `.diff()`. Its difference is I(0).

In [4]:
trending = simulate_brand_data(exog_unit_root=("interest_rate",))
print(check_stationarity(trending.exog))
print()

fixed = trending.exog.assign(interest_rate=trending.exog["interest_rate"].diff()).dropna()
print(check_stationarity(fixed).table[["variable", "order", "role"]].to_string(index=False))

Stationarity check (ADF test, 5% level)
  regression = 'c'

  variable             p (levels)   p (diff)  order  verdict
  -----------------------------------------------------------------------------------------------
  brand_spend               0.000      0.000   I(0)  stationary: use as exog
  pr_reach                  0.000      0.000   I(0)  stationary: use as exog
  sponsorship_reach         0.000      0.000   I(0)  stationary: use as exog
  interest_rate             0.578      0.000   I(1)  trending: endog, or exog after .diff()
  consumer_confidence       0.000      0.000   I(0)  stationary: use as exog

           variable order role
        brand_spend  I(0) exog
           pr_reach  I(0) exog
  sponsorship_reach  I(0) exog
      interest_rate  I(0) exog
consumer_confidence  I(0) exog


### Options and limits

- `regression="ct"` adds a linear trend to the levels test. Use it when a series could be stationary around a straight line. The brand metrics stay I(1) under it.
- `signif` sets the significance level (default 0.05).
- The ADF test has low power against series that revert slowly. In short samples a persistent I(0) series can come back as I(1). Treat the verdict as a guide and overrule it when you know the series.

In [5]:
print(check_stationarity(data.endog, regression="ct"))

Stationarity check (ADF test, 5% level)
  regression = 'ct'

  variable             p (levels)   p (diff)  order  verdict
  -----------------------------------------------------------------------------------------------
  organic_sales             0.400      0.000   I(1)  trending: endog, or exog after .diff()
  brand_awareness           0.512      0.000   I(1)  trending: endog, or exog after .diff()
  brand_consideration       0.242      0.000   I(1)  trending: endog, or exog after .diff()


## 2. `granger_pretest`: what helps predict what?

### What it does

`granger_pretest` fits a VAR in levels on the brand metrics and the drivers together, then runs F tests on the lags. It answers four questions:

| Table | Question |
|---|---|
| Drivers | Does past X help predict the brand metric? |
| Delay | Does past X still help once this week's drivers are known? |
| Links | Does past awareness help predict consideration? And the other pairs |
| Feedback | Does the past brand metric help predict the driver? |

A fifth table, **same period**, shows the correlation of the VAR residuals for each pair. It catches effects inside the week that no lag can see. It cannot say which way they run.

### Why it can run before the rank is chosen

Toda and Yamamoto (1995): fit the VAR with one lag more than the order you test. The extra lag absorbs the unit roots. The F test on the remaining lags is then valid for stationary and cointegrated series alike. The lag order comes from the Hannan-Quinn criterion.

### Why the model needs it

Each table maps to a modelling choice:

- **Drivers** and **links** say which series belong in `exog` and `endog`.
- **Delay** says whether same-week `exog` is enough or the fit needs `exog_lags`.
- **Feedback** checks the assumption that drivers are exogenous. If budgets follow performance, the coefficients in `B` are biased.

The drivers below are brand spend and the two macro series. That matches the setup in `docs/granger_pretest.md`.

In [6]:
drivers = data.exog[["brand_spend", "interest_rate", "consumer_confidence"]]
pre = granger_pretest(data.endog, drivers)
assert isinstance(pre, GrangerPretestResult)
print(pre)

Granger pretest (Toda-Yamamoto)
  lags tested = 1   extra lags = 1   n_obs = 258   alpha = 0.05

Drivers: does past X help predict the brand metric?
  cause -> effect                                      F   p-value  verdict
  --------------------------------------------------------------------------------
  brand_spend -> organic_sales                      0.00     0.966  no clear evidence
  interest_rate -> organic_sales                    0.17     0.681  no clear evidence
  consumer_confidence -> organic_sales              0.00     0.979  no clear evidence
  brand_spend -> brand_awareness                   11.09     0.001  helps predict
  interest_rate -> brand_awareness                  0.95     0.330  no clear evidence
  consumer_confidence -> brand_awareness            0.00     0.964  no clear evidence
  brand_spend -> brand_consideration                1.54     0.216  no clear evidence
  interest_rate -> brand_consideration             10.40     0.001  helps predict
  consumer_c

### Checking it against the truth

The truth is a chain. Spend moves awareness. Awareness moves consideration and consideration moves sales. The macro series move consideration. Nothing feeds back into the drivers and every driver effect lands in the same week.

- **Drivers:** the three true effects are found (p ≤ 0.001). The six absent ones are not.
- **Delay:** nothing. Past spend predicts awareness only because spend persists from week to week. Once this week's spend is in the equation, last week's adds nothing.
- **Links:** awareness to consideration and consideration to sales are found. The four reverse or skipped directions are not.
- **Feedback:** one false positive, consideration to interest rate (p = 0.010). There are nine feedback tests at the 5% level, so one or two false positives are expected. Across 200 seeds each feedback test fires 3% to 7% of the time (`docs/granger_pretest.md`).
- **Same period:** spend and awareness move together within the week (correlation 0.58). So do the macro series and consideration.

### Spotting a delayed effect

With `exog_lag_share=0.5` half of every driver effect arrives a week late. The delay table now picks up the true delayed effects. A clear delay result is the signal to fit with `exog_lags=1`.

In [7]:
lagged = simulate_brand_data(exog_lag_share=0.5)
pre_lagged = granger_pretest(lagged.endog, lagged.exog[drivers.columns])

delay = pre_lagged.delay[["cause", "effect", "f_stat", "p_value"]]
print(delay.round(4).to_string(index=False))

              cause              effect  f_stat  p_value
        brand_spend       organic_sales  0.0300   0.8626
      interest_rate       organic_sales  1.7711   0.1845
consumer_confidence       organic_sales  0.4318   0.5117
        brand_spend     brand_awareness 27.8640   0.0000
      interest_rate     brand_awareness  1.3655   0.2437
consumer_confidence     brand_awareness  0.6045   0.4376
        brand_spend brand_consideration  0.2375   0.6265
      interest_rate brand_consideration 11.7095   0.0007
consumer_confidence brand_consideration 11.5244   0.0008

All three true delayed effects are found (p < 0.001). The six pairs with no effect stay above 0.05.

### The tables as data

Each table is a `DataFrame` for further work. `lags`, `extra_lags` and `n_obs` record what was tested.

In [8]:
print(repr(pre))
pre.tests.head()

GrangerPretestResult(lags=1, extra_lags=1, n_obs=258)


,cause,effect,group,f_stat,df_num,df_den,p_value
0,brand_awareness,organic_sales,metric -> metric,0.121423,1,245,7.277950e-01
1,brand_consideration,organic_sales,metric -> metric,45.138421,1,245,1.275187e-10
2,brand_spend,organic_sales,driver -> metric,0.001808,1,245,9.661207e-01
3,interest_rate,organic_sales,driver -> metric,0.169928,1,245,6.805366e-01
4,consumer_confidence,organic_sales,driver -> metric,0.000673,1,245,9.793288e-01


## 3. What this unlocks

The three pretests now cover every choice the model needs before sampling:

```python
from bayesian_vecm import BayesianVECM, check_stationarity, granger_pretest, select_coint_rank

roles = check_stationarity(df)  # which columns go in endog and exog
pre = granger_pretest(df[roles.endog], df[roles.exog])  # links, delay, feedback
k_ar_diff = 1  # HQ on a levels VAR, see docs/choosing_rank_and_lags.md
rank = select_coint_rank(df[roles.endog], k_ar_diff=k_ar_diff).rank

model = BayesianVECM(k_ar_diff=k_ar_diff, coint_rank=rank, deterministic="co")
model.fit(df[roles.endog], exog=df[roles.exog])
```

The Granger pretest asks its questions of the raw data. Notebook 13 asks the same questions of the fitted posterior, with `granger_causality`. It also splits the forecast uncertainty by source with `fevd`.